In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openrouter import ChatOpenRouter

In [3]:
load_dotenv()

True

In [4]:
llm = ChatOpenRouter(
    model="inclusionai/ling-3.0-flash-vl:free",
    temperature=0
)

In [5]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [9]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {
    "configurable": {
        "thread_id": "1"
    }
}

workflow.invoke({"topic": "Football"}, config=config1)

{'topic': 'Football',
 'joke': 'Here\'s a football joke for you:\n\n**A coach looks at his player and says, "I told you to run with the ball — not the ball with you!"**\n\nThe player replies, "Coach, I\'m just trying to carry my responsibilities." 😄🏈',
 'explanation': '## Explanation of the Joke\n\nThis joke works on **two levels** — a football instruction and a play on words:\n\n### The Setup\nIn football, the correct action is to **"run with the ball"** (i.e., carry the ball and run forward). The coach is correcting the player, implying the player is doing it wrong — perhaps the ball is just sort of "with" them rather than being actively carried and driven forward. The phrase **"the ball with you"** is a humorous inversion of "run with the ball," making it sound like the ball is passively accompanying the player instead of the other way around.\n\n### The Punchline\nThe player responds with **"Coach, I\'m just trying to carry my responsibilities."** This is where the wordplay kicks i

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Here\'s a football joke for you:\n\n**A coach looks at his player and says, "I told you to run with the ball — not the ball with you!"**\n\nThe player replies, "Coach, I\'m just trying to carry my responsibilities." 😄🏈', 'explanation': '## Explanation of the Joke\n\nThis joke works on **two levels** — a football instruction and a play on words:\n\n### The Setup\nIn football, the correct action is to **"run with the ball"** (i.e., carry the ball and run forward). The coach is correcting the player, implying the player is doing it wrong — perhaps the ball is just sort of "with" them rather than being actively carried and driven forward. The phrase **"the ball with you"** is a humorous inversion of "run with the ball," making it sound like the ball is passively accompanying the player instead of the other way around.\n\n### The Punchline\nThe player responds with **"Coach, I\'m just trying to carry my responsibilities."** This is where t

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Here\'s a football joke for you:\n\n**A coach looks at his player and says, "I told you to run with the ball — not the ball with you!"**\n\nThe player replies, "Coach, I\'m just trying to carry my responsibilities." 😄🏈', 'explanation': '## Explanation of the Joke\n\nThis joke works on **two levels** — a football instruction and a play on words:\n\n### The Setup\nIn football, the correct action is to **"run with the ball"** (i.e., carry the ball and run forward). The coach is correcting the player, implying the player is doing it wrong — perhaps the ball is just sort of "with" them rather than being actively carried and driven forward. The phrase **"the ball with you"** is a humorous inversion of "run with the ball," making it sound like the ball is passively accompanying the player instead of the other way around.\n\n### The Punchline\nThe player responds with **"Coach, I\'m just trying to carry my responsibilities."** This is where 

In [17]:
workflow.get_state({
    "configurable": {"thread_id": "1", "checkpoint_id": "1f1b5923-4dff-66d4-8000-af93c80ffc3d"}
})

StateSnapshot(values={'topic': 'Football'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b5923-4dff-66d4-8000-af93c80ffc3d'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-21T07:58:18.214149+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b5923-4dfb-6b84-bfff-e8360a0006db'}}, tasks=(PregelTask(id='cbeadfc5-59ab-e921-a9c8-2a42ecb4025f', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Here\'s a football joke for you:\n\n**A coach looks at his player and says, "I told you to run with the ball — not the ball with you!"**\n\nThe player replies, "Coach, I\'m just trying to carry my responsibilities." 😄🏈'}),), interrupts=())